## Semantic vs. Minimal PLACEHOLDER MD
In all past sections we have seen the same trend, that whatever was the scale/metric, at opposite ends stood `Minimal` and `Semantic` hierarchy, meaning in most cases they seemed to work in a contrary way, where on performed better, the other performed worse and vice versa. Other hierarchies stayed in the middle, they never stood out, neither from the mentioned two hierarchies, nor from each other, i.e. they were rarely distinguishable from each other.

For these reasons, in this last section I compare the performance of these two hierarchies specifically. This allows me to make plots which would have been overwhelming or even unfeasible when dealing with five hierarchies. 

In [1]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root

import pandas as pd
import numpy as np
import plotly.express as px
import python.utils as utils
import python.plots as plots
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS, TimeSeriesType
from python.aggregation import MONTHS, FORTNIGHTS, WEEKS, WEEKDAYS, WEEKENDS, DAYS
from python.temporal_hierarchy import TemporalHierarchy, MinimalTemporalHierarchy, SemanticTemporalHierarchy
from python.error_metrics import ErrorMetrics, MergedErrorMetrics, MetricsIndex, LoadedErrorMetrics, ErrorDifference, FilteredErrorMetrics, ClonedErrorMetrics, CustomMetrics, NO_INDEX
from python.error_metrics import MetricsIndex as MI # Alias
from python.forecast_reconciliation import ForecastReconciliation

/home/egyetem/semester_six/forecast_research/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
em = LoadedErrorMetrics('metrics/base_hier_errors.csv')
inc = pd.read_csv('metrics/base_hier_inconsistency.csv', index_col=0)

sth = SemanticTemporalHierarchy()
mth = MinimalTemporalHierarchy()
wth = TemporalHierarchy([MONTHS, WEEKS, DAYS], 'Minimal+Weeks')
wweth = TemporalHierarchy([MONTHS, WEEKS, WEEKDAYS, WEEKENDS, DAYS], 'Weekends+Weekdays')
wfth = TemporalHierarchy([MONTHS, FORTNIGHTS, WEEKS, DAYS], 'Weeks+Biweeks')

error_metric = 'mase'
em = FilteredErrorMetrics(em, MI(error_metric=error_metric))

diffs = MergedErrorMetrics([
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=sth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=wth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=wweth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=mth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=wfth)),
])

nrmse_em, nrmse_diffs = em, diffs

### QQ plots
First I look at QQ plots comparing the distributions of the reconciled forecasts from the two hierarchies, at daily and monthly granularity separately. In both plots the outliers are filtered out for higher clarity.

In [3]:
plots.plot_error_qq(nrmse_em,
                    MI(hierarchy=mth, forecast_type=RECONCILED_FORECASTS, granularity=DAYS),
                    MI(hierarchy=sth, forecast_type=RECONCILED_FORECASTS, granularity=DAYS),
                    outlier_threshold=0.995).show()

At daily granularity we can see that the first few percentiles are (very slightly) better with the `Minimal` hierarchy, but otherwise the distributions are quite similar, with slight improvements at higher percentiles/errors. We have already seen that the sheer volume of outliers are not smaller with `Minimal` hierarchy, in fact they are slightly bigger, hence why the later percentiles show `Semantic` as better.

In [4]:
plots.plot_error_qq(nrmse_em,
                    MI(hierarchy=mth, forecast_type=RECONCILED_FORECASTS, granularity=MONTHS),
                    MI(hierarchy=sth, forecast_type=RECONCILED_FORECASTS, granularity=MONTHS),
                    outlier_threshold=0.995).show()

At monthly granularity there is no percentile where `Minimal` would be better, and we can see progressively improving percentiles with `Semantic` hierarchy. This suggests a notion that adding hierarchy levels could improve the top level accurarcy, which we will explore later.

Next we look at QQ plots comparing the distributions of error differences.

In [5]:
plots.plot_error_qq(nrmse_diffs,
                    MI(hierarchy=f'{mth} - {mth}', granularity=DAYS),
                    MI(hierarchy=f'{sth} - {mth}', granularity=DAYS),
                    outlier_threshold=0.995).show()

From these plots we can deduce that in cases where there were great improvements to daily granularity errors, `Semantic` performed better, with greater reductions. When the daily error increased, then `Semantic` performed worse, by increasing the errors slightly more than `Minimal`. 

In [6]:
plots.plot_error_qq(nrmse_diffs,
                    MI(hierarchy=f'{mth} - {mth}', granularity=MONTHS),
                    MI(hierarchy=f'{sth} - {mth}', granularity=MONTHS),
                    outlier_threshold=0.995).show()

From the same plot but with monthly granularity we can see that basically each percentile is smaller for `Semantic` hierarchy, meaning it reduced the errors more effectively, and increased them less. This further reinforces that `Semantic` performed better at the monthly granularity level.

### Errors as functions of each other
Here we look at scatter plots of the errors from the two hierarchies. The plots will include the identity line and also an OLS linear estimator trendline. Note that in these plots, we don't treat the errors as independent distributions, but rather as error pairs, so these plots contain much more information.

In [7]:
plots.plot_error_func(nrmse_em,
                      MI(hierarchy=mth, forecast_type=RECONCILED_FORECASTS, granularity=DAYS),
                      MI(hierarchy=sth, forecast_type=RECONCILED_FORECASTS, granularity=DAYS),
                      outlier_threshold=0.95).show()

At daily granularity errors we can see that the OLS trendline shows that `Minimal` performs worse at higher errors, while it performs better at lower errors. However this is only a slight difference, as we can see the coefficient of the trendline is 0.91, and it has a very high R2 value, meaning most samples are very close to this line, and there are only a few (quite visible) outliers.

We have seen that the median (and mean) of `Minimal` hierarchy reconciled errors are lower at daily granularity, so we can deduct that the density of low errors (at which `Minimal` performs better), are much higher than that of higher errors, but this plot is incapable of showing that.

In [8]:
plots.plot_error_func(nrmse_em,
                      MI(hierarchy=mth, forecast_type=RECONCILED_FORECASTS, granularity=MONTHS),
                      MI(hierarchy=sth, forecast_type=RECONCILED_FORECASTS, granularity=MONTHS),
                      outlier_threshold=0.95).show()

At monthly granularity the same plot is much more one sided. The coefficient of the OLS trendline is 0.81 and it still has a very high R2 score (meaning trustable a trendline). This makes it further evident that `Semantic` hierarchy is much better at optimizing for errors at monthly granularity.

As I said we cannot really see the densities/frequencies of errors at different magnitudes in this scatter plot format, so I will make a heatmap of these last two plots. I will make two plots of each granularity, one being a raw heatmap, and the other being a heatmap with log scale coloring.

In [9]:
for log in (False, True):
    plots.plot_error_heatmap(nrmse_em,
                            MI(hierarchy=mth, forecast_type=RECONCILED_FORECASTS, granularity=DAYS),
                            MI(hierarchy=sth, forecast_type=RECONCILED_FORECASTS, granularity=DAYS),
                            outlier_threshold=0.95, bins=75, log_scale=log).show()

From the regular scale coloring we can clearly see how the very small errors were the most frequent, with lesser and lesser number of error pairs propagating in the exact direction of the identity line. There are no other dense spots outside this line whatsoever.

When we move to log scale coloring, we can see more information, it looks a lot more like the scatter plots at this point, but from the coloring we can still see that around the trendline are the most dense parts, with some very light spots around the outliers we have seen on the scatter plots.

In [10]:
for log in (False, True):
    plots.plot_error_heatmap(nrmse_em,
                            MI(hierarchy=mth, forecast_type=RECONCILED_FORECASTS, granularity=MONTHS),
                            MI(hierarchy=sth, forecast_type=RECONCILED_FORECASTS, granularity=MONTHS),
                            outlier_threshold=0.95, bins=75, log_scale=log).show()

These heatmaps at monthly granularity are very similar and we can deduct the same conclusions.

### QQ plot of outliers
In this section I will compare the distribution of just the outliers specifically, on log scale

In [11]:
nrmse_outliers_em = CustomMetrics.keep_outliers(nrmse_em, 0.95, log=True)

In [12]:
plots.plot_error_qq(nrmse_outliers_em,
                    MI(hierarchy=mth, forecast_type=RECONCILED_FORECASTS, granularity=DAYS),
                    MI(hierarchy=sth, forecast_type=RECONCILED_FORECASTS, granularity=DAYS),
                    outlier_threshold=0.999).show()

When comparing the distributions of just the top 5% outliers (per hierarchy) of daily errors, we can see that Minimal has slightly slightly higher low outliers, and slightly smaller big outliers.  

In [13]:
plots.plot_error_qq(nrmse_outliers_em,
                    MI(hierarchy=mth, forecast_type=RECONCILED_FORECASTS, granularity=MONTHS),
                    MI(hierarchy=sth, forecast_type=RECONCILED_FORECASTS, granularity=MONTHS),
                    outlier_threshold=0.999).show()

On monthly granularity there is a much bigger difference, `Semantic` achieves lower outliers, except for the last three percentiles.

### Semantic - Minimal difference 
Next I examine the differences between the reconciled forecasts themselves.

In [14]:
hier_diffs = ErrorDifference(nrmse_em,
                MI(hierarchy=mth, forecast_type=RECONCILED_FORECASTS),
                MI(hierarchy=sth, forecast_type=RECONCILED_FORECASTS))

In [15]:
hier_diffs.stats

count  \
hierarchy          forecast_type                               granularity error_metric method            
Semantic - Minimal reconciled_forecasts - reconciled_forecasts Days        mase         shr     13000.0   
                                                               Months      mase         shr     13000.0   

                                                                                                        mean  \
hierarchy          forecast_type                               granularity error_metric method                 
Semantic - Minimal reconciled_forecasts - reconciled_forecasts Days        mase         shr     3.049935e+07   
                                                               Months      mase         shr     2.448081e+00   

                                                                                                         std  \
hierarchy          forecast_type                               granularity error_metric method                 
Semantic - Minimal reconciled_forecasts - reconciled_forecasts Days        mase         shr     3.477461e+09   
                                                               Months      mase         shr     3.063630e+02   

                                                                                                       min  \
hierarchy          forecast_type                               granularity error_metric method               
Semantic - Minimal reconciled_forecasts - reconciled_forecasts Days        mase         shr     -52.595989   
                                                               Months      mase         shr    -140.932210   

                                                                                                         max  \
hierarchy          forecast_type                               granularity error_metric method                 
Semantic - Minimal reconciled_forecasts - reconciled_forecasts Days        mase         shr     3.964915e+11   
                                                               Months      mase         shr     3.492975e+04   

                                                                                                  median  \
hierarchy          forecast_type                               granularity error_metric method             
Semantic - Minimal reconciled_forecasts - reconciled_forecasts Days        mase         shr    -0.003705   
                                                               Months      mase         shr    -0.019393   

                                                                                                     q10  \
hierarchy          forecast_type                               granularity error_metric method             
Semantic - Minimal reconciled_forecasts - reconciled_forecasts Days        mase         shr    -0.676074   
                                                               Months      mase         shr    -0.733983   

                                                                                                     q25  \
hierarchy          forecast_type                               granularity error_metric method             
Semantic - Minimal reconciled_forecasts - reconciled_forecasts Days        mase         shr    -0.147106   
                                                               Months      mase         shr    -0.193797   

                                                                                                     q75  \
hierarchy          forecast_type                               granularity error_metric method             
Semantic - Minimal reconciled_forecasts - reconciled_forecasts Days        mase         shr     0.040937   
                                                               Months      mase         shr     0.025770   

                                                                                                     q90  
hierarchy          forecast_type          

From the statistics of the error differences, it is visible that the median is negative, meaning in more than half the cases, `Semantic` achieved lower errors at both granularities. We can see lower quantiles at monthly granularity, suggesting that `Semantic` was better at improving the monthly error.

In [16]:
hier_diffs.get_error_stats(MI(granularity=DAYS),
                           df_filter=lambda df: df[df < np.quantile(df.values.flatten(), 0.9999)]
)

,,,,,count,mean,std,min,max,median,q10,q25,q75,q90
hierarchy,forecast_type,granularity,error_metric,method,,,,,,,,,,
Semantic - Minimal,reconciled_forecasts - reconciled_forecasts,Days,mase,shr,12998.0,-0.136461,1.778579,-52.595989,85.670836,-0.003711,-0.676081,-0.14715,0.040869,0.172757


After looking at the stats with filtered out extreme outliers, we can see that the mean of difference at daily granularity is negative too, not just the median. But it must be mentioned that both the mean and median is extremely small (in absolute value), so `Minimal` is not lacking too much

In [17]:
hier_diffs.get_error_stats(MI(granularity=MONTHS),
                           df_filter=lambda df: df[df < np.quantile(df.values.flatten(), 0.9999)]
)

,,,,,count,mean,std,min,max,median,q10,q25,q75,q90
hierarchy,forecast_type,granularity,error_metric,method,,,,,,,,,,
Semantic - Minimal,reconciled_forecasts - reconciled_forecasts,Months,mase,shr,12999.0,-0.238841,2.043146,-140.93221,112.822754,-0.019397,-0.733991,-0.193824,0.025755,0.149561


At monthly granularity, the absolute value of the mean of differences is much more significant, further suggesting that `Semantic` hierarchy is better at producing reliable monthly forecasts.

With this outlier filtering, let us look back at the plain statistics of the reconciled forecast errors of both hierarchies, to make the mean and variance statistics more reliable.

In [18]:
nrmse_em.get_error_stats(MI(granularity=DAYS, hierarchy=[sth, mth], forecast_type=RECONCILED_FORECASTS),
                           df_filter=lambda df: df[df < np.quantile(df.values.flatten(), 0.99)]
)

,,,,,count,mean,std,min,max,median,q10,q25,q75,q90
hierarchy,forecast_type,granularity,error_metric,method,,,,,,,,,,
Semantic,reconciled_forecasts,Days,mase,shr,12846.0,1.859055,2.325224,0.001558,15.451622,0.953049,0.294888,0.606948,1.909854,5.090649
Minimal,reconciled_forecasts,Days,mase,shr,12819.0,1.942651,2.502948,0.001330,15.386244,0.956809,0.275188,0.605474,1.890541,5.559890


From the daily error filtered statistics, we can still see that `Minimal` achieved slightly lower median error (with other qunatiles going back and forth, as we have already seen from the QQ plots), but it in fact achieved worse mean error, by a slight difference. The variances of the errors are very similar, with slighly higher one for `Minimal`

In [19]:
nrmse_em.get_error_stats(MI(granularity=MONTHS, hierarchy=[sth, mth], forecast_type=RECONCILED_FORECASTS),
                           df_filter=lambda df: df[df < np.quantile(df.values.flatten(), 0.99)]
)

,,,,,count,mean,std,min,max,median,q10,q25,q75,q90
hierarchy,forecast_type,granularity,error_metric,method,,,,,,,,,,
Semantic,reconciled_forecasts,Months,mase,shr,12918.0,1.483994,2.093371,0.000102,15.444914,0.731388,0.094307,0.277857,1.719121,3.792936
Minimal,reconciled_forecasts,Months,mase,shr,12862.0,1.627567,2.360239,0.000095,15.414498,0.761267,0.099077,0.303561,1.790353,4.327652


At monthly granularity, `Semantic` achieves lower median error, lower mean error and lower variance (by a much bigger margin). All other percentiles favor `Semantic` also, as we have already seen in the QQ plots.

Next I will examine some plots of the error differences between the two reconciled errors

In [20]:
plots.plot_error_diff(nrmse_em,
                    MI(hierarchy=mth, forecast_type=RECONCILED_FORECASTS, granularity=DAYS),
                    MI(hierarchy=sth, forecast_type=RECONCILED_FORECASTS, granularity=DAYS), outlier_threshold=0.999)

From this plot we see that when `Minimal` achieved high daily errors, there were no cases when `Semantic` did significantly worse. The cases where this had happened, are only at lower error values.

In [21]:
plots.plot_error_diff(nrmse_em,
                    MI(hierarchy=mth, forecast_type=RECONCILED_FORECASTS, granularity=MONTHS),
                    MI(hierarchy=sth, forecast_type=RECONCILED_FORECASTS, granularity=MONTHS), outlier_threshold=0.999)

At monthly granularity, the trend is much more apparent, with `Semantic` making big improvements in many cases, and only worsening predictions in a low absolute value.

Next I look at a scatter plot of the differences are daily and monthly granularity

In [22]:
plots.plot_error_func(hier_diffs, MI(granularity=DAYS), MI(granularity=MONTHS), outlier_threshold=0.999,)

Visually, the magnitude of differences are much higher towards the negative values, meaning there were move extreme improvements than worsenings. Other than that, we can see the dashed identity line, and how most dots are below that, meaning monthly difference was lower than the daily.

In [23]:
plots.plot_error_distribution(hier_diffs, [MI(granularity=DAYS), MI(granularity=MONTHS)], outlier_threshold=0.99)

From the histogram of the error differences at daily and monthly granularity, we can see how there were many cases of slightly positive daily differences (meaning worse predictions by `Semantic`), with more big monthly improvements at the negative end.

Next, let's plot the sign distribution of the differences

In [24]:
plots.plot_error_signs(hier_diffs, MI(granularity=DAYS), MI(granularity=MONTHS), percentage=True)

It is apparent that in almost half the cases, `Semantic` decreased the error at both granularities, while at a third of the cases, it increased both erros. There were three times as much cases when monthly error decreased while daily increased, than there were when daily error decreased, while monthly increased.

Just over 50% of cases showed improvements to daily granularity errors, while almost two thirds of cases showed improvements to monthly granularity errors when going from `Minimal` to `Semantic` hierarchy.

Next I will plot the error differences and the base forecast inconsistencies as a scatter plot.

In [25]:
plots.plot_error_over_inconsistency(hier_diffs, MI(granularity=DAYS), inc, outlier_threshold=0.99)

These plots look very similar to when I plotted the error differences to base forecasts, in function of the base inconsistencies. The only pattern that emerges, is that the greatest improvements with `Semantic` hierarchy could be made when the base inconsistency itself was small.

In [26]:
plots.plot_error_over_inconsistency(hier_diffs, MI(granularity=MONTHS), inc, outlier_threshold=0.99)

The same can be said at monthly hierarchy.